# Cost

> Completion cost calculation

In [ ]:
#| default_exp cost

## Imports

In [ ]:
#| export
import nbdev, yaml, json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import *
from fastcore.net import urljson

from fastllm.types import *
from fastllm.normalize import *
from fastllm.streaming import *
from fastllm.acomplete import *

In [ ]:
#| hide
import yaml,json,base64,litellm,httpx
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms
from fastspec.spec import *
from fastspec.oapi import *

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
repo_path = nbdev.config.get_config().config_path; repo_path
specs_path = repo_path/'specs'

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
import json
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

## Cost Metadata

In [ ]:
models = [
    ('claude-sonnet-4-20250514', {}),
    ('gpt-4o-mini', {}),
    ('models/gemini-3-flash-preview', {}),
    ('accounts/fireworks/models/kimi-k2p5', dict(vendor_name='fireworks_ai')),
]
mtok = 1024
resps = [await acomplete([Msg('user', content=[Part('text', "Say 'hello' in French.")])], model=name, max_tokens=mtok, **kw) for name, kw in models]

In [ ]:
resps[-1]

Completion(model='accounts/fireworks/models/kimi-k2p5', message=Msg(role='assistant', content=[Part(type='thinking', text='The user wants me to say "hello" in French. The word for "hello" in French is "Bonjour". \n\nThis is a simple, straightforward request. I should provide the translation and perhaps a bit of context (like when it\'s used), but the main thing is to say "Bonjour".\n\nI should not overcomplicate this. Just provide the French word for hello.', data=None), Part(type='text', text='Bonjour.', data={'citations': []})], data=None), finish_reason='stop', usage=Usage(prompt_tokens=15, completion_tokens=83, total_tokens=98, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 15, 'total_tokens': 98, 'completion_tokens': 83, 'prompt_tokens_details': {'cached_tokens': 0}}), tool_calls=[], api_name=<api_name.openai_chat: 'openai_chat'>, vendor_name='fireworks_ai', raw={'id': 'chatcmpl-aec592e8609a4d89ae0990062e6238da', 'object': 'chat.completion', 'c

In [ ]:
L(resps).attrgot('model')

['claude-sonnet-4-20250514', 'gpt-4o-mini-2024-07-18', 'models/gemini-3-flash-preview', 'accounts/fireworks/models/kimi-k2p5']

In [ ]:
list(L(resps).attrgot('usage'))

[Usage(prompt_tokens=15, completion_tokens=8, total_tokens=23, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 15, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 8, 'service_tier': 'standard', 'inference_geo': 'not_available'}),
 Usage(prompt_tokens=14, completion_tokens=10, total_tokens=24, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 14, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 10, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 24}),
 Usage(prompt_tokens=8, completion_tokens=2, total_tokens=98, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=88, raw={'promptTokenCount': 8, 'candidatesTokenCount': 2, 'totalTokenCount': 98, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}], 'thoughtsTokenCount': 88}),
 Usage(prompt_tokens=15, completi

In [ ]:
list(L(resps).attrgot('vendor_name'))

[<api_name.anthropic: 'anthropic'>,
 <api_name.openai: 'openai'>,
 <api_name.gemini: 'gemini'>,
 'fireworks_ai']

Now that we have a fully functional fastllm library let's now look how we can calculate cost of a given Completion

In [ ]:
infer_api_name??


```python
def infer_api_name(model):
    "Infer api_name from model"
    if "claude" in model: return ApiName.anthropic
    if "gemini" in model: return ApiName.gemini
    if any(o in model for o in ('gpt','o3-','o4-')): return ApiName.openai
```

**File:** `~/aai-ws/fastllm/fastllm/acomplete.py`

In [ ]:
#| export
model_prices_url = 'https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json'
@flexicache(time_policy(24*60*60))
def model_prices_meta(): return urljson(model_prices_url)

In [ ]:
mp = model_prices_meta()

In [ ]:
L(mp.keys())[:10]

['sample_spec', '1024-x-1024/50-steps/bedrock/amazon.nova-canvas-v1:0', '1024-x-1024/50-steps/stability.stable-diffusion-xl-v1', '1024-x-1024/dall-e-2', '1024-x-1024/max-steps/stability.stable-diffusion-xl-v1', '256-x-256/dall-e-2', '512-x-512/50-steps/stability.stable-diffusion-xl-v0', '512-x-512/dall-e-2', '512-x-512/max-steps/stability.stable-diffusion-xl-v0', 'ai21.j2-mid-v1']

We can try exact match first. For gemini models we can replace `models/` with `gemini/`. For others we can use `third_party_name` which can be prepended and try an exact match. If nothing matches we can do substring match and get the first match.

In [ ]:
resp = litellm.completion(
    model="fireworks_ai/accounts/fireworks/models/kimi-k2p5",  # openai-compatible prefix
    messages=[{"role":"user","content":"Say hello in French"}],
    max_tokens=64,
)
cost = litellm.completion_cost(completion_response=resp)
print(f"Cost: ${cost}")

Cost: $0.0001972


In [ ]:
#| export
@flexicache(time_policy(24*60*60))
def get_model_meta(model, vendor_name=None, tfm=noop):
    "Look up cost metadata for `model` from litellm price map, using `vendor_name` prefix if needed."
    vendor_name = ifnone(vendor_name, infer_api_name(model))
    mp = model_prices_meta()
    if model in mp: key = model
    elif vendor_name=='gemini' and model.startswith('models/'): key = f"gemini/{model.removeprefix('models/')}"
    elif vendor_name:                                           key = f"{vendor_name}/{model}"
    return dict2obj(tfm(mp.get(key), model, vendor_name))

In [ ]:
#| export
codex_pricing = {
    "input_cost_per_token": 0.10 / 1_000_000,
    "cache_creation_input_token_cost": 0.10 / 1_000_000,
    "cache_read_input_token_cost": 0.10 / 1_000_000,
    "output_cost_per_token": 0.50 / 1_000_000,
}

def get_model_info(mn, vendor_name=None):
    info = get_model_meta(mn, vendor_name)
    # anthropic web search 
    if 'search_context_cost_per_query' in info:
        info['supports_web_search'] = True
    # add reasoning to kimi
    if 'kimi' in mn: 
        if 'k2p6' in mn: info = get_model_meta(mn.replace('k2p6', 'k2p5'), vendor_name)
        info['supports_reasoning'] = True
        info['supports_vision'] = True
    # add web search to gpt
    if mn in ("gpt-5.4", "gpt-5.4-mini"):
        info['supports_web_search'] = True
        info.pop('mode', None)
    # codex pricing
    if vendor_name == 'codex': info = merge(info, codex_pricing)
    return dict2obj(info)

In [ ]:
test_cases = [
    ('claude-sonnet-4-20250514', 'anthropic'),
    ('gpt-4o-mini', 'openai'),
    ('models/gemini-3-flash-preview', 'gemini'),
    ('accounts/fireworks/models/kimi-k2p5', 'fireworks_ai'),
]
for m, v in test_cases:
    r = get_model_meta(m, v)
    if r: print(f"✅ {m:50s} in=${r.get('input_cost_per_token',0):.8f}  out=${r.get('output_cost_per_token',0):.8f}")
    else: print(f"❌ {m:50s} NOT FOUND")

✅ claude-sonnet-4-20250514                           in=$0.00000300  out=$0.00001500
✅ gpt-4o-mini                                        in=$0.00000015  out=$0.00000060
✅ models/gemini-3-flash-preview                      in=$0.00000050  out=$0.00000300
✅ accounts/fireworks/models/kimi-k2p5                in=$0.00000060  out=$0.00000300


## Calculate Cost

In [ ]:
list(ApiName)

[<api_name.openai: 'openai'>,
 <api_name.openai_chat: 'openai_chat'>,
 <api_name.anthropic: 'anthropic'>,
 <api_name.gemini: 'gemini'>]

Now we are ready to write helpers for calculating costs for each api

In [ ]:
sonnet_meta = get_model_meta('claude-sonnet-4-20250514', 'anthropic')
gpt4o_meta = get_model_meta('gpt-4o-mini', 'openai')
gemini_flash_meta   = get_model_meta('models/gemini-3-flash-preview', 'gemini')
kimi_fireworks_meta = get_model_meta('accounts/fireworks/models/kimi-k2p5', 'fireworks_ai')

In [ ]:
def filter_cost_meta(d): return {k:v for k,v in d.items() if 'cost' in k}

In [ ]:
print(filter_cost_meta(sonnet_meta), '\n')
print(filter_cost_meta(gpt4o_meta), '\n')
print(filter_cost_meta(gemini_flash_meta), '\n')
print(filter_cost_meta(kimi_fireworks_meta), '\n')

{'cache_creation_input_token_cost': 3.75e-06, 'cache_creation_input_token_cost_above_1hr': 6e-06, 'cache_read_input_token_cost': 3e-07, 'input_cost_per_token': 3e-06, 'input_cost_per_token_above_200k_tokens': 6e-06, 'output_cost_per_token_above_200k_tokens': 2.25e-05, 'cache_creation_input_token_cost_above_200k_tokens': 7.5e-06, 'cache_read_input_token_cost_above_200k_tokens': 6e-07, 'output_cost_per_token': 1.5e-05, 'search_context_cost_per_query': {'search_context_size_high': 0.01, 'search_context_size_low': 0.01, 'search_context_size_medium': 0.01}} 

{'cache_read_input_token_cost': 7.5e-08, 'cache_read_input_token_cost_priority': 1.25e-07, 'input_cost_per_token': 1.5e-07, 'input_cost_per_token_batches': 7.5e-08, 'input_cost_per_token_priority': 2.5e-07, 'output_cost_per_token': 6e-07, 'output_cost_per_token_batches': 3e-07, 'output_cost_per_token_priority': 1e-06} 

{'cache_read_input_token_cost': 5e-08, 'input_cost_per_audio_token': 1e-06, 'input_cost_per_token': 5e-07, 'output_co

In [ ]:
filter_cost_meta(kimi_fireworks_meta)

{'cache_read_input_token_cost': 1e-07,
 'input_cost_per_token': 6e-07,
 'output_cost_per_token': 3e-06}

### OpenAI Chat

In [ ]:
comps = dict(L(resps).map(lambda o: (o.model,o)))

In [ ]:
# comps

In [ ]:
dict(L(resps).map(lambda o: (o.model,o.usage.raw)))

{'claude-sonnet-4-20250514': {'input_tokens': 15,
  'cache_creation_input_tokens': 0,
  'cache_read_input_tokens': 0,
  'cache_creation': {'ephemeral_5m_input_tokens': 0,
   'ephemeral_1h_input_tokens': 0},
  'output_tokens': 8,
  'service_tier': 'standard',
  'inference_geo': 'not_available'},
 'gpt-4o-mini-2024-07-18': {'input_tokens': 14,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 10,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 24},
 'models/gemini-3-flash-preview': {'promptTokenCount': 8,
  'candidatesTokenCount': 2,
  'totalTokenCount': 98,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}],
  'thoughtsTokenCount': 88},
 'accounts/fireworks/models/kimi-k2p5': {'prompt_tokens': 15,
  'total_tokens': 98,
  'completion_tokens': 83,
  'prompt_tokens_details': {'cached_tokens': 0}}}

In [ ]:
def openai_chat_cost(usage, m):
    raw = usage.raw
    pd, cd = raw.get('prompt_tokens_details', {}), raw.get('completion_tokens_details', {})
    cached = pd.get('cached_tokens', 0)
    in_audio, out_audio = pd.get('audio_tokens', 0), cd.get('audio_tokens', 0)
    in_txt  = raw['prompt_tokens']     - cached - in_audio
    out_txt = raw['completion_tokens'] - out_audio
    cost  = in_txt  * m.input_cost_per_token  + out_txt * m.output_cost_per_token
    cost += cached  * m.get('cache_read_input_token_cost', 0)
    cost += in_audio  * m.get('input_cost_per_audio_token', 0)
    cost += out_audio * m.get('output_cost_per_audio_token', 0)
    return cost

In [ ]:
#| export
@patch(as_prop=True)
def cost(self:Completion):
    meta = dict2obj(get_model_info(self.model, self.vendor_name))
    if self.api_name == ApiName.openai_chat: return openai_chat_cost(self.usage, meta)
    if self.api_name == ApiName.openai:      return openai_responses_cost(self.usage, meta)
    if self.api_name == ApiName.gemini:      return gemini_cost(self.usage, meta)
    if self.api_name == ApiName.anthropic:   return anthropic_cost(self.usage, meta)

In [ ]:
litecomp = litellm.completion(model="fireworks_ai/accounts/fireworks/models/kimi-k2p5", messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
litellm_cost = litellm.completion_cost(completion_response=litecomp); litellm_cost

0.0001937

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='accounts/fireworks/models/kimi-k2p5', vendor_name='fireworks_ai', max_tokens=64, temperature=0.0)
comp.cost

0.0001937

In [ ]:
litecomp.choices[0].message.content

'The user wants me to say "hello" in French. This is a simple request. The most common way to say hello in French is "Bonjour". \n\nI could also mention other variations:\n- "Salut" (informal, hi/hey)\n- "Bonsoir" (good evening)\n-'

In [ ]:
comp.message.content[0].text

'The user wants me to say "hello" in French. This is a simple request. The most common way to say hello in French is "Bonjour". \n\nI could also mention other variations:\n- "Salut" (informal, hi/hey)\n- "Bonsoir" (good evening)\n-'

In [ ]:
test_eq(litellm_cost, comp.cost)

In [ ]:
mn = 'accounts/fireworks/models/kimi-k2p5'
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=mn, vendor_name='fireworks_ai', max_tokens=64, temperature=1.0)
litecomp = litellm.completion(model=f"fireworks_ai/{mn}", messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=1.0)
litellm_cost = litellm.completion_cost(completion_response=litecomp)
test_eq(litellm_cost, comp.cost)

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

In [ ]:
comp = await acomplete([Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])], model='gpt-4o-audio-preview', api_name=ApiName.openai_chat, vendor_name='openai_chat', temperature=1.0)

In [ ]:
get_model_meta('gpt-4o-mini')

```python
{ 'cache_read_input_token_cost': 7.5e-08,
  'cache_read_input_token_cost_priority': 1.25e-07,
  'input_cost_per_token': 1.5e-07,
  'input_cost_per_token_batches': 7.5e-08,
  'input_cost_per_token_priority': 2.5e-07,
  'litellm_provider': 'openai',
  'max_input_tokens': 128000,
  'max_output_tokens': 16384,
  'max_tokens': 16384,
  'mode': 'chat',
  'output_cost_per_token': 6e-07,
  'output_cost_per_token_batches': 3e-07,
  'output_cost_per_token_priority': 1e-06,
  'supports_function_calling': True,
  'supports_parallel_function_calling': True,
  'supports_pdf_input': True,
  'supports_prompt_caching': True,
  'supports_response_schema': True,
  'supports_service_tier': True,
  'supports_system_messages': True,
  'supports_tool_choice': True,
  'supports_vision': True}
```

In [ ]:
get_model_meta('gpt-4o-audio-preview')

```python
{ 'input_cost_per_audio_token': 4e-05,
  'input_cost_per_token': 2.5e-06,
  'litellm_provider': 'openai',
  'max_input_tokens': 128000,
  'max_output_tokens': 16384,
  'max_tokens': 16384,
  'mode': 'chat',
  'output_cost_per_audio_token': 8e-05,
  'output_cost_per_token': 1e-05,
  'supports_audio_input': True,
  'supports_audio_output': True,
  'supports_function_calling': True,
  'supports_parallel_function_calling': True,
  'supports_system_messages': True,
  'supports_tool_choice': True}
```

In [ ]:
comp.usage.raw

{'prompt_tokens': 86,
 'completion_tokens': 33,
 'total_tokens': 119,
 'prompt_tokens_details': {'cached_tokens': 0,
  'audio_tokens': 69,
  'text_tokens': 17,
  'image_tokens': 0},
 'completion_tokens_details': {'reasoning_tokens': 0,
  'audio_tokens': 0,
  'accepted_prediction_tokens': 0,
  'rejected_prediction_tokens': 0,
  'text_tokens': 33}}

In [ ]:
#| export
def openai_chat_cost(usage, m):
    raw = usage.raw
    pd, cd = raw.get('prompt_tokens_details', {}), raw.get('completion_tokens_details', {})
    cached = pd.get('cached_tokens', 0)
    in_audio, out_audio = pd.get('audio_tokens', 0), cd.get('audio_tokens', 0)
    in_txt  = raw['prompt_tokens']     - cached - in_audio
    out_txt = raw['completion_tokens'] - out_audio
    cost  = in_txt  * m.input_cost_per_token  + out_txt * m.output_cost_per_token
    cost += cached  * m.get('cache_read_input_token_cost', 0)
    cost += in_audio  * m.get('input_cost_per_audio_token', 0)
    cost += out_audio * m.get('output_cost_per_audio_token', 0)
    return cost

In [ ]:
comp.cost

0.0031325000000000003

In [ ]:
litecomp = litellm.completion(
    model="gpt-4o-audio-preview",
    messages=[{"role":"user","content":[
        {"type":"input_audio","input_audio":{"data":base64.b64encode(wav_data).decode(),"format":"wav"}},
        {"type":"text","text":"What is this audio saying?"}
    ]}],
    temperature=1.0
)
litellm.completion_cost(completion_response=litecomp), comp.cost

(0.0031325000000000003, 0.0031325000000000003)

In [ ]:
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

### OpenAI Repsonses

In [ ]:
#| export
def openai_responses_cost(usage, m):
    raw = usage.raw
    cached = raw.get('input_tokens_details', {}).get('cached_tokens', 0)
    in_txt = raw['input_tokens'] - cached
    cost  = in_txt * m.input_cost_per_token + raw['output_tokens'] * m.output_cost_per_token
    cost += cached * m.get('cache_read_input_token_cost', 0)
    return cost

#### Text

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='gpt-4o-mini', max_tokens=64, temperature=0.0, api_name=ApiName.openai)
litecomp = litellm.responses(model="gpt-4o-mini", input=[{"type":"message","role":"user","content":[{"type":"input_text","text":"Say hello in French"}]}], max_output_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='gpt-4o-mini', max_tokens=64, temperature=0.0, api_name=ApiName.openai, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# o-series models on Responses API use `max_output_tokens`, not `max_tokens`
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='o4-mini', api_name=ApiName.openai, temperature=1.0, max_tokens=64)
litecomp = litellm.responses(model="o4-mini", input=[{"type":"message","role":"user","content":[{"type":"input_text","text":"Say hello in French"}]}], temperature=1.0, max_output_tokens=64)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='o4-mini', api_name=ApiName.openai, temperature=1.0, max_tokens=64, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

#### Image

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])
comp = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0, api_name=ApiName.openai)

In [ ]:
litecomp = litellm.responses(model="gpt-4o-mini",input=[{"type":"message","role":"user","content":[{"type":"input_image","image_url":img_url},{"type":"input_text","text":"What is this image?"}]}], max_output_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0, api_name=ApiName.openai, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-5) # stream=True bypasses cachy so it's a fresh api call which can differ slightly

### Anthropic

In [ ]:
#| export
def anthropic_cost(usage, m):
    raw = usage.raw
    in_tok = raw['input_tokens']
    cache_read = raw.get('cache_read_input_tokens', 0)
    cc = raw.get('cache_creation', {}) or {}
    cache_5m  = cc.get('ephemeral_5m_input_tokens', 0)
    cache_1h  = cc.get('ephemeral_1h_input_tokens', 0)
    cost  = in_tok     * m.input_cost_per_token
    cost += raw['output_tokens'] * m.output_cost_per_token
    cost += cache_read * m.get('cache_read_input_token_cost', 0)
    cost += cache_5m   * m.get('cache_creation_input_token_cost', 0)
    cost += cache_1h   * m.get('cache_creation_input_token_cost_above_1hr', 0)
    return cost

#### Text:

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

#### Image:

In [ ]:
msg = Msg('user', content=[Part(PartType.input_image, img_url), Part('text', 'What is this image?')])
comp = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0)
# litellm anthropic image form:
litecomp = litellm.completion(model="anthropic/claude-sonnet-4-20250514",
    messages=[{"role":"user","content":[{"type":"image_url","image_url":{"url":img_url}},{"type":"text","text":"What is this image?"}]}],
    max_tokens=64, temperature=0.0)
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

#### Cache read and write:

In [ ]:
cc = {"cache_control": {"type": "ephemeral"}}
big_text = 'The quick brown fox jumps over the lazy dog. ' * 200
msg1 = Msg('user', content=[Part('text', big_text, data=cc), Part('text', 'Summarize')])
comp1 = await acomplete([msg1], model='claude-sonnet-4-20250514', max_tokens=64)  # writes cache
msg3 = Msg('user', content=[Part('text', 'Now in French')])
comp2 = await acomplete([msg1, comp1.message, msg3], model='claude-sonnet-4-20250514', max_tokens=64)  # reads cache

In [ ]:
big_msgs1 = [{"role":"user","content":[
    {"type":"text","text":big_text,"cache_control":{"type":"ephemeral"}},
    {"type":"text","text":"Summarize"}]}]
litecomp1 = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=big_msgs1, max_tokens=64)

big_msgs2 = big_msgs1 + [
    {"role":"assistant","content":litecomp1.choices[0].message.content},
    {"role":"user","content":"Now in French"}]
litecomp2 = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=big_msgs2, max_tokens=64)

We hit cache when we make the first api call with litellm. So testing only the second api call cost is enough:

In [ ]:
test_eq(comp2.cost, litellm.completion_cost(completion_response=litecomp2))

In [ ]:
# Streaming - as a sanity check
it1 = await acomplete([msg1], model='claude-sonnet-4-20250514', max_tokens=64, stream=True)  # writes cache
it2 = await acomplete([msg1, comp1.message, msg3], model='claude-sonnet-4-20250514', max_tokens=64, stream=True)  # reads cache
async for comp1 in it1: pass
async for comp2 in it2: pass

We hit cache when we make the first api call with litellm. So testing only the second api call cost is enough:

In [ ]:
test_eq(comp2.cost, litellm.completion_cost(completion_response=litecomp2))

### Gemini

In [ ]:
#| export
def gemini_cost(usage, m):
    raw = usage.raw
    prompt_tot = raw.get('promptTokenCount', 0)
    tier = '_above_200k_tokens' if prompt_tot > 200_000 else ''
    in_rate    = m.get(f'input_cost_per_token{tier}')       or m.input_cost_per_token
    out_rate   = m.get(f'output_cost_per_token{tier}')      or m.output_cost_per_token
    cache_rate = m.get(f'cache_read_input_token_cost{tier}')or m.get('cache_read_input_token_cost', 0)
    audio_rate = m.get('input_cost_per_audio_token')  # None if not priced separately

    cached = raw.get('cachedContentTokenCount', 0)
    # Gemini 3 Pro supports bills audio at the standard input rate (no separate input_cost_per_audio_token key in the metadata)
    audio  = sum(d['tokenCount'] for d in raw.get('promptTokensDetails', []) if d.get('modality')=='AUDIO') if audio_rate else 0
    in_txt = prompt_tot - cached - audio

    thoughts = raw.get('thoughtsTokenCount', 0) or 0
    cands    = raw.get('candidatesTokenCount', 0) or 0
    reason_rate = m.get('output_cost_per_reasoning_token') or out_rate

    cost  = in_txt * in_rate + cands * out_rate
    cost += cached * cache_rate
    cost += audio  * (audio_rate or 0)
    cost += thoughts * reason_rate
    return cost

#### Basic call with flash/pro 3:

In [ ]:
flash_mn, pro_mn   = 'models/gemini-3-flash-preview', 'models/gemini-3-pro-preview'
lflash_mn, lpro_mn = 'gemini/gemini-3-flash-preview', 'gemini/gemini-3-pro-preview'

There is a slight difference in request payloads, fastllm uses the original request param field names from the spec which is camel case, and litellm normalizes them. Gemini api accepts both:

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=flash_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lflash_mn, messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=flash_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=pro_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=pro_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

#### Thinking with flash/pro 3:

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=flash_mn, reasoning_effort='low', temperature=0.0)
litecomp = litellm.completion(model=lflash_mn, messages=[{"role":"user","content":"What is 123 * 456?"}], reasoning_effort='low', temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=flash_mn, reasoning_effort='low', temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3) # 0.02 cents difference due to 67 more thought tokens

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=pro_mn, reasoning_effort='low', temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":"What is 123 * 456?"}], reasoning_effort='low', temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=pro_mn, reasoning_effort='low', temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3) # 0.05 cents difference

#### Audio with flash/pro 3:

In [ ]:
msg = Msg('user', content=[Part(PartType.input_audio, audio_b64), Part('text', 'What is this audio saying?')])
comp = await acomplete([msg], model=flash_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lflash_mn,
    messages=[{"role":"user","content":[{"type":"input_audio","input_audio":{"data":base64.b64encode(wav_data).decode(),"format":"wav"}},{"type":"text","text":"What is this audio saying?"}]}],
    max_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model=flash_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

JSON payload key differences (camel case vs snake in `inlineData`) result in no cache hit and different completions, so we use `test_close` here:

In [ ]:
msg = Msg('user', content=[Part(PartType.input_audio, audio_b64), Part('text', 'What is this audio saying?')])
comp = await acomplete([msg], model=pro_mn, temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":[{"type":"input_audio","input_audio":{"data":audio_b64.split(',', 1)[1],"format":"wav"}},{"type":"text","text":"What is this audio saying?"}]}], temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3) # 0.03 cents difference
litellm.completion_cost(completion_response=litecomp), comp.cost

(0.005738, 0.006085999999999999)

In [ ]:
it = await acomplete([msg], model=pro_mn, temperature=0.0, stream=True)
async for comp in it: pass
litellm.completion_cost(completion_response=litecomp), comp.cost

(0.005738, 0.005018)

The larger gap here isn't a costing bug — it's that the camelCase `inlineData` payload misses cachy, so each call hits the API fresh and Gemini Pro 3 generates a different response (more thought tokens + more output tokens). Cost functions are still correct given each response's `usage.raw`.

#### Caching with flash/pro 3:

In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 250
msg1 = Msg('user', content=[Part('text', big_text + "\n\nWrite sentence that is repeated")])
comp1 = await acomplete([msg1], model=flash_mn, temperature=0.0)

In [ ]:
msg3 = Msg('user', content=[Part('text', 'Now in French!')])
comp2 = await acomplete([msg1, comp1.message, msg3], model=flash_mn, temperature=0.0)  # should read
print('cached:', comp2.usage.raw.get('cachedContentTokenCount', 0))

cached: 0


In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 250
big_msgs1 = [{"role":"user","content":big_text + "\n\nWrite sentence that is repeated"}]
litecomp1 = litellm.completion(model=lflash_mn, messages=big_msgs1, temperature=0.0)

In [ ]:
big_msgs2 = big_msgs1 + [{"role":"assistant","content":litecomp1.choices[0].message.content}, {"role":"user","content":"Now in French!"}]
litecomp2 = litellm.completion(model=lflash_mn, messages=big_msgs2, temperature=0.0)

Another difference between fastllm and litellm is the url construction, litellm uses `https://generativelanguage.googleapis.com/{v1alpha|v1beta}/models/{model}:generateContent?key={API_KEY}` and in fastllm we use spec's base_url which is `https://generativelanguage.googleapis.com/` and endpoints come directly from the spec, e.g. `v1beta/...`, also the api key is sent as a header

Gemini implicit caching is not guaranteed, so it is particularly difficult to test deterministically:

In [ ]:
litellm.completion_cost(completion_response=litecomp2), comp2.cost

(0.002403, 0.002403)

In [ ]:
comp2.usage.raw

{'promptTokenCount': 2532,
 'candidatesTokenCount': 28,
 'totalTokenCount': 2911,
 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2532}],
 'thoughtsTokenCount': 351}

In [ ]:
litecomp2.usage

Usage(completion_tokens=379, prompt_tokens=2532, total_tokens=2911, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=351, rejected_prediction_tokens=None, text_tokens=28, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=None, text_tokens=2532, image_tokens=None, video_tokens=None), cache_read_input_tokens=None)

#### Large context with flash/pro 3:

In [ ]:
from litellm.types.utils import ModelResponse, Usage as LUsage, PromptTokensDetailsWrapper, CompletionTokensDetailsWrapper

In [ ]:
fake_usage = Usage(prompt_tokens=250_000, completion_tokens=1000, total_tokens=251_000,
                   raw={'promptTokenCount':250_000, 'candidatesTokenCount':1000, 'totalTokenCount':251_000})
pro_meta = dict2obj(get_model_meta('models/gemini-3-pro-preview', 'gemini'))
cost = gemini_cost(fake_usage, pro_meta)
expected = 250_000 * 4e-06 + 1000 * 1.8e-05  # above-200k rates
test_close(cost, expected, 1e-10)

In [ ]:
lusage = LUsage(
    prompt_tokens=250_000, completion_tokens=1000, total_tokens=251_000,
    prompt_tokens_details=PromptTokensDetailsWrapper(text_tokens=250_000),
    completion_tokens_details=CompletionTokensDetailsWrapper(text_tokens=1000, reasoning_tokens=0),
)
fake_resp = ModelResponse(model='gemini-3-pro-preview', usage=lusage)
fake_resp._hidden_params = {'custom_llm_provider': 'gemini'}

In [ ]:
test_eq(cost,litellm.completion_cost(completion_response=fake_resp))

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()